In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
from tqdm import tqdm
import os

os.environ["NIXTLA_ID_AS_COL"] = "False"

In [2]:
# Load data files
data = pd.read_csv("C:/Data/M5store.csv")

In [3]:
# Define the MASE function
def mase(y, y_pred, y_train, seasonality=1):
    """
    Calculate Mean Absolute Scaled Error (MASE)
    y: Actual values
    y_pred: Predicted values
    y_train: Training data for scaling factor
    seasonality: Seasonal period for naive forecasting
    """
    mae = np.mean(np.abs(y - y_pred))
    naive_forecast_errors = np.abs(y_train[seasonality:] - y_train[:-seasonality])
    scaling_factor = np.mean(naive_forecast_errors)
    return mae / scaling_factor

# Leave-One-Out Cross-Validation for last m points
def leave_one_out_cv_last_m_with_naive(df, m, h, model, seasonality=7):
    """
    Perform Leave-One-Out Cross-Validation on the last m data points with both AutoETS and naive forecasts.
    
    df: DataFrame with columns 'ds', 'y', and 'unique_id'.
    m: Number of data points from the end of the dataset for cross-validation.
    h: Number of steps ahead for forecasting.
    model: Model object with .fit() and .predict() methods.
    seasonality: Seasonal period for naive forecasting.
    
    Returns:
    - MASE values for AutoETS and naive forecasts.
    """
    errors_autoets = []  # Store actual and predicted values for AutoETS
    errors_naive = []    # Store actual and predicted values for naive forecast
    start_index = len(df) - m  # Start index for cross-validation

    for i in range(start_index, len(df) - h + 1):
        # Training and test split
        train_subset = df.iloc[:i]  # Use all points up to the current fold
        test_subset = df.iloc[i:i + h]

        # Fit the model on the training subset
        autoets = model.fit(train_subset['y'].values)

        # Predict for the test subset using AutoETS
        y_hat_autoets = autoets.predict(h=h).get("mean")
        errors_autoets.extend(zip(test_subset['y'].values, y_hat_autoets))

        # Calculate naive forecast
        y_hat_naive = train_subset['y'].iloc[-seasonality:].values.tolist() * h
        y_hat_naive = y_hat_naive[:h]  # Ensure forecast length matches h
        errors_naive.extend(zip(test_subset['y'].values, y_hat_naive))

    # Calculate MASE for AutoETS
    actual_autoets = np.array([e[0] for e in errors_autoets])
    predicted_autoets = np.array([e[1] for e in errors_autoets])
    train_series = df['y'].values  # Full training series for scaling
    mase_autoets = mase(actual_autoets, predicted_autoets, train_series)

    # Calculate MASE for naive forecast
    actual_naive = np.array([e[0] for e in errors_naive])
    predicted_naive = np.array([e[1] for e in errors_naive])
    mase_naive = mase(actual_naive, predicted_naive, train_series)

    return mase_autoets, mase_naive

In [ ]:
# Parameters
unique_store_ids = data['store_id'].unique()
m = 28  # Only consider the last m points for cross-validation
h = 1  # Number of steps ahead for forecasting
seasonality = 7  # Weekly seasonality for daily data

results = []

# Loop through each store_id with progress tracking
for store_id in tqdm(unique_store_ids, desc="Processing all store_id series"):
    df = data.loc[data['store_id'] == store_id, ['d', 'revenue', 'store_id']]
    df = df.rename(columns={'d': 'ds', 'revenue': 'y', 'store_id': 'unique_id'})
    
    model = AutoETS(model=["Z", "Z", "Z"], alias="AutoETS", damped=True, season_length=seasonality)
    mase_autoets, mase_naive = leave_one_out_cv_last_m_with_naive(df, m, h, model, seasonality)
    results.append({"store_id": store_id, "AutoETS_MASE": mase_autoets, "Naive_MASE": mase_naive})

# Create a summary table
summary_table = pd.DataFrame(results)

# Calculate average MASE for both AutoETS and naive forecasts
average_autoets_mase = summary_table["AutoETS_MASE"].mean()
average_naive_mase = summary_table["Naive_MASE"].mean()

print("Summary Table of MASE for Each Series:")
print(summary_table)
print(f"\nAverage AutoETS MASE across all series: {average_autoets_mase}")
print(f"Average Naive MASE across all series: {average_naive_mase}")

Processing all store_id series:  80%|████████████████████████████████████████▊          | 8/10 [01:42<00:26, 13.27s/it]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110})

# Prepare data: parse dates and ensure ds/y columns
plot_data = data.copy()
plot_data['ds'] = pd.to_datetime(plot_data['d'])
plot_data['y'] = plot_data['revenue']
store_ids = sorted(plot_data['store_id'].unique())
pal = plt.cm.tab10(np.linspace(0, 1, len(store_ids)))

# ── Easy: MASE Comparison — AutoETS vs Naive by Store ─────────────────────────
df_plot = summary_table[summary_table['store_id'] != 'Overall Average'].copy()
x = np.arange(len(df_plot)); width = 0.38
fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width/2, df_plot['AutoETS_MASE'], width, label='AutoETS', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, df_plot['Naive_MASE'], width, label='Naive Seasonal', color='#95a5a6', edgecolor='white')
ax.axhline(1.0, color='red', lw=1.8, linestyle='--', label='MASE = 1 (= Naive baseline)')
ax.axhline(summary_table.loc[summary_table['store_id']=='Overall Average','AutoETS_MASE'].values[0],
           color='#3498db', lw=1.5, linestyle=':', alpha=0.7, label='AutoETS avg')
for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=8.5, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(df_plot['store_id'], fontsize=11)
ax.set_ylabel('MASE (lower = better)'); ax.legend(fontsize=11)
ax.set_title('AutoETS vs Naive Forecast — MASE by Store\n(MASE < 1 means better than naive weekly seasonal)',
             fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# ── Medium: Revenue Time Series — Last 60 Days per Store ─────────────────────
fig, axes = plt.subplots(5, 2, figsize=(15, 18))
for ax, store, color in zip(axes.flat, store_ids, pal):
    sd = plot_data[plot_data['store_id'] == store].sort_values('ds').tail(90)
    ax.plot(sd['ds'], sd['y'], lw=1.8, color=color)
    ax.fill_between(sd['ds'], sd['y'], alpha=0.15, color=color)
    ax.set_title(f'Store {store}', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30); ax.tick_params(labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
plt.suptitle('Daily Revenue — Last 90 Days per Store', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# ── Medium: Day-of-Week Seasonality Across All Stores ─────────────────────────
plot_data['dow'] = plot_data['ds'].dt.dayofweek
dow_avg = plot_data.groupby(['store_id', 'dow'])['y'].mean().reset_index()
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(12, 6))
for store, color in zip(store_ids, pal):
    sd = dow_avg[dow_avg['store_id'] == store]
    ax.plot(sd['dow'], sd['y'], 'o-', lw=2, color=color, markersize=7, label=store)
ax.set_xticks(range(7)); ax.set_xticklabels(dow_names, fontsize=11)
ax.set_ylabel('Average Daily Revenue ($)', fontsize=12)
ax.set_title('Day-of-Week Seasonality — Average Revenue per Store',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2); ax.spines[['top','right']].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout(); plt.show()

# ── Hard: Revenue Heatmap — Store × Month ────────────────────────────────────
plot_data['month'] = plot_data['ds'].dt.to_period('M').astype(str)
monthly = plot_data.groupby(['store_id', 'month'])['y'].sum().reset_index()
heat_df = monthly.pivot(index='store_id', columns='month', values='y')
heat_df_norm = heat_df.div(heat_df.mean(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(heat_df_norm, ax=ax, cmap='RdYlGn', center=1.0, linewidths=0.3,
            linecolor='white', cbar_kws={'label': 'Revenue / Store Monthly Mean'})
ax.set_title('Monthly Revenue Heatmap — Normalized by Store Mean\n(Green = above average, Red = below average)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Store')
ax.tick_params(axis='x', rotation=45, labelsize=8)
plt.tight_layout(); plt.show()

# ── Hard: Rolling 28-Day Revenue Trend — All Stores on One Plot ──────────────
fig, ax = plt.subplots(figsize=(16, 7), facecolor='#0f0f1a')
ax.set_facecolor('#0f0f1a')
for store, color in zip(store_ids, pal):
    sd = plot_data[plot_data['store_id']==store].sort_values('ds').set_index('ds')['y']
    roll = sd.rolling(28, min_periods=7).mean()
    ax.plot(roll.index, roll.values, lw=2, color=color, label=store, alpha=0.9)
ax.set_xlabel('Date', fontsize=12, color='white')
ax.set_ylabel('28-Day Rolling Average Revenue ($)', fontsize=12, color='white')
ax.set_title('Rolling 28-Day Revenue Trend — All Stores\n(Smoothed to reveal seasonal patterns)',
             fontsize=13, fontweight='bold', color='white')
ax.legend(fontsize=9, ncol=2, facecolor='#1a1a2e', labelcolor='white', edgecolor='grey')
ax.tick_params(colors='white')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
for sp in ax.spines.values(): sp.set_color('#333')
plt.tight_layout(); plt.show()